# DPE global weighting rules and neural combiners — Kaggle / Colab Run All

**No attached files or code-cell edits are required.** Select a **GPU runtime**, enable Internet on Kaggle, import this notebook and choose **Run All**. It discovers or downloads official Waterbirds, trains a separate 100-epoch ERM backbone and 15 DPE heads for each missing seed (0, 1, 2), then fits and freezes the 30 global-rule/neural configurations on the pooled B+C training set (479 images).

Existing H2 source ZIPs, a previous combined study ZIP with `source_archives/`, or completed runs are optional and save training time. The default `MODE="auto"` trains missing seeds; `MODE="cached"` keeps the earlier cache-only behavior. Neural-combiner fitting runs on CPU; the backbones need real GPU time.

`EVALUATE_TEST=True` evaluates every configuration separately on test after all configurations across all seeds have been fitted and frozen. Set it to False for a training-only run. This is a project adaptation inspired by **Regularized Neural Ensemblers (AutoML 2025)**; previously inspected test comparisons remain exploratory.

Keep WORK_DIR persistent to survive VM deletion. New backbones resume at completed epoch boundaries; completed backbones, features, heads and neural selections are reused after checks. The compact download includes source heads/prediction bundles but excludes raw images, backbone/resume checkpoints and raw feature arrays.


In [ ]:
MODE = "auto"       # Prepare/train missing seeds; "cached" requires existing H2 sources.
SOURCE = None       # Optional H2 source ZIP/directory or previous combined study ZIP.
SEEDS = [0, 1, 2]    # Explicitly set [0] for a one-seed pilot.
TRACK = "standard"
EVALUATE_TEST = True
CONFIG_PATH = "configs/dpe_head_reweighting/combined_comparison.yaml"
DEVICE = "auto"     # Prefer GPU for new backbones. Neural combiners run on CPU.
WORKERS = None      # Defaults to 2; use 0 if worker processes are unavailable.
DATA_DIR = None     # Discover attached Waterbirds or download automatically.
REPO_PATH = None
WORK_DIR = None
SEARCH_PATHS = None
REPO_URL = "https://github.com/chautanphat/waterbirds-shortcut-mitigation.git"
REPO_REF = "experiment/dpe-neural-combiner"  # Published implementation branch.


In [ ]:
import importlib.util
import json
import os
from pathlib import Path
import subprocess
import sys

working = Path.cwd().resolve()
PLATFORM = "kaggle" if Path("/kaggle/working").is_dir() else "colab" if Path("/content").is_dir() else "local"
def compatible(path):
    marker = path / "src/methods/dpe_head_reweighting/neural_study.py"
    return marker.is_file() and "NEURAL_STUDY_RUNTIME_VERSION = 4" in marker.read_text()
candidates = [working, *working.parents, working / "waterbirds-shortcut-mitigation"]
REPO = Path(REPO_PATH).expanduser().resolve() if REPO_PATH else next((p for p in candidates if compatible(p)), None)
if WORK_DIR is None:
    base = Path("/kaggle/working") if PLATFORM == "kaggle" else Path("/content") if PLATFORM == "colab" else (REPO / "outputs" if REPO else working)
    WORK_DIR = base / "dpe_neural_combiners_session"
WORK_DIR = Path(WORK_DIR).expanduser().resolve()
WORK_DIR.mkdir(parents=True, exist_ok=True)
if REPO is None:
    REPO = WORK_DIR / "repository-neural-v4"
    if not REPO.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--single-branch", "--branch", REPO_REF, REPO_URL, str(REPO)], check=True)
if not compatible(REPO):
    raise FileNotFoundError("Use the updated neural study runtime (version 2); set REPO_PATH to the published implementation branch.")
missing = [package for module, package in [("numpy", "numpy>=1.26,<3"), ("scipy", "scipy>=1.11,<2"),
    ("torch", "torch>=2.6,<3"), ("torchvision", "torchvision>=0.21,<1"),
    ("yaml", "PyYAML>=6,<7"), ("pandas", "pandas>=2.2,<3"),
    ("PIL", "Pillow>=10.3,<13"), ("tqdm", "tqdm")] if importlib.util.find_spec(module) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", *missing], check=True)
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
from IPython.display import display, FileLink
from src.methods.dpe_head_reweighting.neural_study import run_study
print("Platform:", PLATFORM, "| Repository:", REPO, "| Workspace:", WORK_DIR)


## Fit on B+C and evaluate every configuration

The fixed YAML declares 400 full-batch CPU float64 AdamW steps, group-balanced BCE, weight decay 0.01/0.001 and prediction dropout 0/0.25. There are 30 configurations: one shared uniform reference, 12 power rules, 2 softmax rules, 2 top-k rules, 3 joint fits and 10 neural configurations. Every configuration fits on B+C. Global rules use pooled per-head WGA or pooled worst-group NLL; neural combiners use group-balanced BCE. No configuration is replaced by a family winner. Both weight-decay values are separate configurations. There is no C selection, uniform fallback or early stopping; all 30 configurations are evaluated when test evaluation is enabled.

Auto mode prepares Waterbirds once and trains only missing seeds using the unchanged H2 recipe. Optional input bundles are read only. Cached mode fails before training if a requested seed is missing. Every completed seed produces a source ZIP and an updated partial study ZIP. Re-running verifies saved model states and choices. Changed code/configuration/input identities produce a separate output directory.


In [ ]:
if SEARCH_PATHS is None:
    SEARCH_PATHS = [REPO / "outputs/dpe_head_reweighting_runs"]
    if PLATFORM == "kaggle":
        SEARCH_PATHS += [Path("/kaggle/input")]
    elif PLATFORM == "colab":
        SEARCH_PATHS += list(Path("/content").glob("*.zip")) + [Path("/content/uploads")]
    else:
        SEARCH_PATHS += list(working.glob("*.zip"))
STUDY = run_study(WORK_DIR, source=SOURCE, search_roots=SEARCH_PATHS, track=TRACK,
                  seeds=SEEDS, config_path=CONFIG_PATH, evaluate_test=EVALUATE_TEST,
                  mode=MODE, device=DEVICE, workers=WORKERS, data_dir=DATA_DIR)
output = Path(STUDY["output_dir"])
for seed in SEEDS:
    selection = json.loads((output / "comparisons" / f"seed{seed}" / "selection.json").read_text())
    print("Seed", seed, "protocol:", selection.get("protocol", "b_select_c"))
    print("Frozen configurations:", [r["id"] for r in selection["candidates"]])
    print("Training complementarity (label-aware oracle is not a predictor):", selection["complementarity"])
    print((output / "comparisons" / f"seed{seed}" / "candidate_comparison.csv").read_text())
if STUDY["summary"] is None:
    print("Configurations saved. Test evaluation was not requested. Set EVALUATE_TEST=True and rerun to evaluate.")
else:
    print(json.dumps(STUDY["summary"], indent=2))


## Download models, source bundles and reports

The ZIP updates after each frozen/evaluated seed, retaining the fixed plan, model states, all configurations and reusable H2 source ZIPs in `source_archives/`. Selection-only source bundles contain B/C predictions; test prediction caches are added only when test evaluation is requested. Final evaluation reports per-group accuracy/counts, paired deltas, changed errors and gate concentration.

Large backbone/resume checkpoints, dataset images and raw features stay in WORK_DIR. Keep that directory to resume training or generate missing test predictions later; the compact ZIP does not contain those large files. The download cell also finds the most recent partial ZIP after an interrupted study.


In [ ]:
if "STUDY" in globals():
    archive = Path(STUDY["archive"])
else:
    partial = sorted((WORK_DIR / "studies").glob("*.zip"), key=lambda p: p.stat().st_mtime)
    if not partial:
        raise FileNotFoundError("No seed has frozen yet. Rerun the study cell to resume saved backbone epochs.")
    archive = partial[-1]
previous = Path.cwd()
try:
    os.chdir(working)
    display(FileLink(os.path.relpath(archive, working), result_html_prefix="Download neural comparison: "))
finally:
    os.chdir(previous)
if PLATFORM == "colab":
    from google.colab import files
    files.download(str(archive))
